In [1]:
import pandas as pd
import numpy as np
import sys
from pathlib import Path

# Agregar la raíz del proyecto al path para importar módulos
project_root = Path.cwd().parent if Path.cwd().name == 'gradient_bosting' else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# Importar el pipeline de producción
from src.utils.preprocessing import build_production_pipeline

print("✅ Pipeline de producción importado correctamente")
print("   Este es el MISMO pipeline que usa la API en tiempo real")


✅ Pipeline de producción importado correctamente
   Este es el MISMO pipeline que usa la API en tiempo real


In [3]:
from sklearn.model_selection import train_test_split

# ============================================
# CARGA DE DATASETS Y PREPARACIÓN
# ============================================

# Cargar CSV limpio (el que usa la API en producción)
df_model = pd.read_csv('data/processed/consumo_granada_cleaned.csv')
print(f"📊 Dataset cargado: {df_model.shape}")
print(f"   Columnas: {list(df_model.columns)}")

# Eliminar filas con valores faltantes críticos
df_model = df_model.dropna(subset=['timestamp', 'zone_name', 'temperature', 'consumption_kwh'])
print(f"📊 Después de eliminar NaNs: {df_model.shape}")

# SEPARAR FEATURES y TARGET
X = df_model[['timestamp', 'zone_name', 'temperature']].copy()
y = df_model['consumption_kwh'].copy()

# SEPARAR en Train/Test (80/20)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

print(f"\n✅ Datos preparados:")
print(f"   X_train: {X_train.shape}")
print(f"   X_test: {X_test.shape}")
print(f"   y_train: {y_train.shape}")
print(f"   y_test: {y_test.shape}")


📊 Dataset cargado: (1947461, 7)
   Columnas: ['timestamp', 'zone_id', 'zone_name', 'consumption_kwh', 'temperature', 'fecha', 'hora']
📊 Después de eliminar NaNs: (1947461, 7)

✅ Datos preparados:
   X_train: (1557968, 3)
   X_test: (389493, 3)
   y_train: (1557968,)
   y_test: (389493,)


In [4]:
from sklearn.model_selection import cross_validate
from sklearn.ensemble import GradientBoostingRegressor

# ============================================
# DEFINIR Y ENTRENAR EL MODELO
# ============================================

# Parámetros óptimos del modelo
n_estimators_optimo = 200
learning_rate_optimo = 0.03
max_depth_optimo = 7
min_samples_leaf_optimo = 3
subsample_optimo = 0.8
max_features_optimo = 'sqrt'

# Crear el modelo
gb_model = GradientBoostingRegressor(
    random_state=42,
    n_estimators=n_estimators_optimo,
    learning_rate=learning_rate_optimo,
    max_depth=max_depth_optimo,
    min_samples_leaf=min_samples_leaf_optimo,
    subsample=subsample_optimo,
    max_features=max_features_optimo
)

# Construir el pipeline completo (mismo que producción)
pipeline = build_production_pipeline(gb_model)

print("🔧 Entrenando pipeline completo con 5-Fold Cross-Validation...")
print(f"   Pipeline: ProductionDataCleaner -> FeatureEngineer -> Preprocessing -> GradientBoosting")

# VALIDACIÓN CRUZADA para evaluar el rendimiento
scoring_metrics = ['neg_mean_absolute_error', 'neg_root_mean_squared_error']

cv_results = cross_validate(
    pipeline, 
    X_train, 
    y_train, 
    cv=5,                 
    scoring=scoring_metrics, 
    n_jobs=-1             
)

# PROCESAR LOS RESULTADOS
mae_promedio = -1 * cv_results['test_neg_mean_absolute_error'].mean()
rmse_promedio = -1 * cv_results['test_neg_root_mean_squared_error'].mean()

print(f"\n📈 Resultados Cross-Validation (5-Fold):")
print(f"   MAE Promedio:  {mae_promedio:.2f} kWh")
print(f"   RMSE Promedio: {rmse_promedio:.2f} kWh")

# Comparación con el Objetivo del Proyecto
objetivo_rmse = 342.82

print("\n--- Auditoría del Proyecto ---")
if rmse_promedio <= objetivo_rmse:
    print(f"✅ ¡OBJETIVO RMSE CUMPLIDO! ({rmse_promedio:.2f} <= {objetivo_rmse})")
else:
    print(f"⚠️ Diferencia con objetivo: {rmse_promedio - objetivo_rmse:.2f} kWh")

# ENTRENAR PIPELINE FINAL con todo el training set
print("\n🔧 Entrenando pipeline final con todos los datos de entrenamiento...")
pipeline.fit(X_train, y_train)
print("✅ Pipeline final entrenado y listo para evaluación")


🔧 Entrenando pipeline completo con 5-Fold Cross-Validation...
   Pipeline: ProductionDataCleaner -> FeatureEngineer -> Preprocessing -> GradientBoosting

📈 Resultados Cross-Validation (5-Fold):
   MAE Promedio:  241.45 kWh
   RMSE Promedio: 365.20 kWh

--- Auditoría del Proyecto ---
⚠️ Diferencia con objetivo: 22.38 kWh

🔧 Entrenando pipeline final con todos los datos de entrenamiento...
✅ Pipeline final entrenado y listo para evaluación


In [5]:
from sklearn.metrics import mean_absolute_error, mean_squared_error

# ============================================
# EVALUACIÓN EN EL TEST SET
# ============================================

print("🧪 Evaluando en el test set...")

# Realizar predicción
y_pred = pipeline.predict(X_test)

# Calcular métricas
mae_test = mean_absolute_error(y_test, y_pred)
rmse_test = np.sqrt(mean_squared_error(y_test, y_pred))

print(f"\n📊 Métricas en Test Set:")
print(f"   MAE:  {mae_test:.2f} kWh")
print(f"   RMSE: {rmse_test:.2f} kWh")

# Verificar número de features que genera el pipeline
print(f"\n🔍 Verificación del pipeline:")
X_sample = X_test.head(1)
X_transformed = pipeline[:-1].transform(X_sample)
print(f"   Input: {X_sample.shape} (timestamp, zone_name, temperature)")
print(f"   Después del pipeline: {X_transformed.shape} features")
print(f"   El modelo espera: {gb_model.n_features_in_} features")

if X_transformed.shape[1] == gb_model.n_features_in_:
    print("   ✅ ¡Número de features correcto!")
else:
    print(f"   ⚠️ Mismatch: {X_transformed.shape[1]} vs {gb_model.n_features_in_}")

print("\n✅ Evaluación completada")


🧪 Evaluando en el test set...

📊 Métricas en Test Set:
   MAE:  239.04 kWh
   RMSE: 361.86 kWh

🔍 Verificación del pipeline:
   Input: (1, 3) (timestamp, zone_name, temperature)
   Después del pipeline: (1, 35) features
   El modelo espera: 35 features
   ✅ ¡Número de features correcto!

✅ Evaluación completada


In [6]:
import os
import joblib

# ============================================
# GUARDAR SOLO EL MODELO (NO EL PIPELINE)
# ============================================

# Crear el directorio de modelos si no existe
os.makedirs("data/models", exist_ok=True)

# Extraer el modelo entrenado del pipeline
trained_model = pipeline.named_steps['model']

# Guardar SOLO el modelo (no todo el pipeline)
model_path = "data/models/gradient_boosting_model.joblib"
joblib.dump(trained_model, model_path)

print(f"\n💾 Modelo Gradient Boosting guardado en: {model_path}")
print(f"   Tamaño: {os.path.getsize(model_path) / (1024*1024):.2f} MB")
print(f"   Features esperadas: {trained_model.n_features_in_}")

print("\n✅ Modelo listo para producción")
print("   IMPORTANTE: El modelo espera recibir datos preprocesados por el pipeline de producción")
print("   La API cargará este modelo y lo envolverá en el mismo pipeline usado aquí")



💾 Modelo Gradient Boosting guardado en: data/models/gradient_boosting_model.joblib
   Tamaño: 3.54 MB
   Features esperadas: 35

✅ Modelo listo para producción
   IMPORTANTE: El modelo espera recibir datos preprocesados por el pipeline de producción
   La API cargará este modelo y lo envolverá en el mismo pipeline usado aquí
